In [2]:
import pandas as pd
import numpy as np
import ast
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from shared import load_data, evaluate_model

# 1. LOAD DATA
train_df, test_df, restaurants_df = load_data()

# --- UNIFIED MAPPINGS ---
user_ids = train_df['user_id'].unique()
user_to_idx = {user: idx for idx, user in enumerate(user_ids)}
idx_to_user = {idx: user for user, idx in user_to_idx.items()}

# Mapping item indices perfectly to the restaurants_df order for BOTH models
item_to_idx = pd.Series(restaurants_df.index, index=restaurants_df['business_id']).to_dict()
idx_to_item = {idx: item for item, idx in item_to_idx.items()}
all_businesses = restaurants_df['business_id'].tolist()

num_users = len(user_ids)
num_items = len(restaurants_df)


# ==========================================
# PART 1: COLLABORATIVE FILTERING SETUP (SVD)
# ==========================================
print("Setting up Collaborative Filtering...")
R = np.zeros((num_users, num_items))

for row in train_df.itertuples():
    u_idx = user_to_idx[row.user_id]
    if row.business_id in item_to_idx:
        i_idx = item_to_idx[row.business_id]
        R[u_idx, i_idx] = row.stars

R_df = pd.DataFrame(R)
R_df.replace(0, np.nan, inplace=True)
global_mean = train_df['stars'].mean()

# Imputation
user_means_series = R_df.mean(axis=1, skipna=True)
R_df = R_df.apply(lambda row: row.fillna(user_means_series[row.name]), axis=1)
item_means = R_df.mean(skipna=True).fillna(global_mean)
R_df = R_df.apply(lambda col: col.fillna(item_means[col.name]))
R_df.fillna(global_mean, inplace=True)

R_im = R_df.values

# SVD Generation
k = 4
U, s, Vt = svds(R_im, k=k)
idx_svd = np.argsort(s)[::-1]
U, s, Vt = U[:, idx_svd], s[idx_svd], Vt[idx_svd, :]
predicted_ratings_cf = U @ np.diag(s) @ Vt


# ==========================================
# PART 2: CONTENT-BASED SETUP (META + TEXT)
# ==========================================
print("Setting up Content-Based Meta Profiles...")
BOOLEAN_ATTRIBUTES = {'RestaurantsTakeOut': 'TakeOut', 'OutdoorSeating': 'Outdoor', 'RestaurantsDelivery': 'Deliv', 'GoodForKids': 'GFK'}
CATEGORICAL_ATTRIBUTES = {'RestaurantsPriceRange2': 'Price', 'Ambience': 'Amb'}
ALL_ATTRIBUTES = {**BOOLEAN_ATTRIBUTES, **CATEGORICAL_ATTRIBUTES}

def parse_attributes(attr_str):
    if pd.isna(attr_str): return {}
    return ast.literal_eval(attr_str)

def get_attr(row, attr_name):
    attrs = row.get('parsed_attributes', {})
    if not isinstance(attrs, dict): return 'Unknown'
    return str(attrs.get(attr_name, 'Unknown')).replace("u'", "").replace("'", "")

restaurants_df['parsed_attributes'] = restaurants_df['attributes'].apply(parse_attributes)

for attributes, clean_name in ALL_ATTRIBUTES.items():
    restaurants_df[clean_name] = restaurants_df.apply(lambda row: get_attr(row, attributes), axis=1)

binary_feature_cols = []
for col in list(BOOLEAN_ATTRIBUTES.values()):
    bin_name = col + '_Bin'
    restaurants_df[bin_name] = restaurants_df[col].map({'True': 1, 'False': 0, 'Unknown': 0, 'None': 0}).fillna(0).astype(int)
    binary_feature_cols.append(bin_name)

categorical_clean_names = list(CATEGORICAL_ATTRIBUTES.values())
attr_dummies = pd.get_dummies(restaurants_df[categorical_clean_names], dtype=int) if categorical_clean_names else pd.DataFrame(index=restaurants_df.index)

restaurants_df['cat_list'] = restaurants_df['categories'].astype(str).apply(lambda x: [c.strip().lower() for c in x.split(',')])
categories_dummies = pd.get_dummies(restaurants_df['cat_list'].explode(), dtype=int).groupby(level=0).sum()

feature_df = pd.concat([
    (restaurants_df[binary_feature_cols] * 0.30),
    (attr_dummies * 0.50),
    (categories_dummies * 0.20)
], axis=1)
feature_matrix_normalized = normalize(feature_df.values, norm='l2', axis=1)

print("Setting up Content-Based Text Profiles (SentenceTransformers)...")
K_REVIEWS = 15
MIN_WORDS = 5

quality_reviews = train_df.dropna(subset=['text']).copy()
quality_reviews['datetime'] = pd.to_datetime(quality_reviews['datetime'])
quality_reviews['word_count'] = quality_reviews['text'].str.split().str.len()
quality_reviews = quality_reviews[quality_reviews['word_count'] >= MIN_WORDS]
quality_reviews = quality_reviews.sort_values(['business_id', 'datetime'], ascending=[True, False])

top_k_grouped = quality_reviews.groupby('business_id').head(K_REVIEWS)
reviews_final = top_k_grouped.groupby('business_id')['text'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
reviews_final.rename(columns={'text': 'concat_review'}, inplace=True)

restaurants_df = restaurants_df.merge(reviews_final, on='business_id', how='left')
restaurants_df['concat_review'] = restaurants_df['concat_review'].fillna("")

texts_to_encode = restaurants_df['concat_review'].apply(lambda x: ' '.join(x.split()[:380])).tolist()
model = SentenceTransformer('all-mpnet-base-v2')
text_embeddings = model.encode(texts_to_encode, show_progress_bar=True)
text_matrix_normalized = normalize(text_embeddings, norm='l2', axis=1)

print("Building User Profiles...")
all_train = train_df[train_df['stars'] >= 0.0]
user_meta_profiles = {}
user_text_profiles = {}

for user, group in all_train.groupby('user_id'):
    liked_item_ids = group['business_id'].tolist()
    liked_indices = [item_to_idx[biz] for biz in liked_item_ids if biz in item_to_idx]

    if liked_indices:
        user_meta_profiles[user] = np.asarray(feature_matrix_normalized[liked_indices].mean(axis=0))
        user_text_profiles[user] = np.asarray(text_matrix_normalized[liked_indices].mean(axis=0))


# ==========================================
# PART 3: HARD SWITCHING PREDICTION LOOP
# ==========================================
print("Generating Switching Hybrid Predictions...")
gamma = 0.25 # CONTENT-BASED INNER WEIGHT: 25% Text, 75% Meta
SWITCH_THRESHOLD = 5 # Users with < 5 ratings get CBF, >= 5 get SVD

test_users = test_df['user_id'].unique()
predictions = {}

for user in test_users:

    # 1. Determine user history size and indices
    if user in user_to_idx:
        u_idx = user_to_idx[user]
        num_ratings = np.count_nonzero(R[u_idx, :])
        already_rated_indices = np.where(R[u_idx, :] > 0)[0]
    else:
        u_idx = -1
        num_ratings = 0
        already_rated_indices = []

    # 2. Hard Switch Logic
    if num_ratings < SWITCH_THRESHOLD:
        # --> SWITCH TO 100% CONTENT-BASED <--
        if user in user_meta_profiles and user in user_text_profiles:
            sim_meta = cosine_similarity(user_meta_profiles[user].reshape(1, -1), feature_matrix_normalized).flatten()
            sim_text = cosine_similarity(user_text_profiles[user].reshape(1, -1), text_matrix_normalized).flatten()
            final_scores = (gamma * sim_text) + ((1 - gamma) * sim_meta)
        else:
            final_scores = np.zeros(num_items)

    else:
        # --> SWITCH TO 100% COLLABORATIVE FILTERING <--
        final_scores = predicted_ratings_cf[u_idx, :].copy()

    # 3. Mask already rated items
    if len(already_rated_indices) > 0:
        final_scores[already_rated_indices] = -999.0

    # 4. Extract Top 30 Rankings
    if sum(final_scores) > -999.0 * len(final_scores): # Ensure it isn't all masked/empty
        top_indices = final_scores.argsort()[-30:][::-1]
        predictions[user] = [idx_to_item[i] for i in top_indices]
    else:
        predictions[user] = []

# ==========================================
# PART 4: EVALUATION
# ==========================================
metrics = evaluate_model(predictions, test_df)

results_df = pd.DataFrame([metrics]).round(4)
results_df.index = [f'Switching Hybrid (Threshold={SWITCH_THRESHOLD})']
display(results_df)

Setting up Collaborative Filtering...
Setting up Content-Based Meta Profiles...
Setting up Content-Based Text Profiles (SentenceTransformers)...


Batches: 100%|██████████| 24/24 [02:49<00:00,  7.06s/it]


Building User Profiles...
Generating Switching Hybrid Predictions...


,Hit@10,Hit@20,Hit@30,NDCG@10,NDCG@20,NDCG@30
Switching Hybrid (Threshold=5),0.0712,0.1171,0.1566,0.0364,0.0479,0.0563
